# Layer 1—Intent Agent Fine-Tuning MARBERT,4-class

## 1. Setup & Install

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn statsmodels
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%matplotlib inline
import os, re, json, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                              confusion_matrix, classification_report)
from statsmodels.stats.proportion import proportion_confint
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments, EarlyStoppingCallback)

sns.set_theme(style="whitegrid")
print("Imports OK. CUDA available:", torch.cuda.is_available())


## 2. Config

In [ ]:
SEED = 42
N_SEEDS = 3
SEEDS = [42, 43, 44]
MAX_LENGTH = 128

MODEL_CHECKPOINT = "UBC-NLP/MARBERTv2" 

DATASETS_BASE = "/content/drive/MyDrive/Datasets"
LAYER1_DIR    = f"{DATASETS_BASE}/layer1"
OSACT_DIR     = f"{DATASETS_BASE}/OSACT"

LAYER1_TRAIN_PATH = f"{LAYER1_DIR}/layer1_train.csv"
LAYER1_VAL_PATH   = f"{LAYER1_DIR}/layer1_validation.csv"
LAYER1_TEST_PATH  = f"{LAYER1_DIR}/layer1_test.csv"
DEV_OSACT_PATH    = f"{OSACT_DIR}/dev_osact.csv"   

OUT_DIR      = f"{LAYER1_DIR}/outputs/intent_agent_rebuild"
MODEL_OUT    = f"{OUT_DIR}/model"           
REPORT_DIR   = f"{OUT_DIR}/reports"
for d in [OUT_DIR, MODEL_OUT, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 3. Load Data & Derive Intent Labels

In [ ]:
ID2LABEL = {0: "safe", 1: "hate_speech", 2: "prompt_injection", 3: "sensitive_request"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

def load_intent_split(path, split_name):
    df = pd.read_csv(path)
    before = len(df)
    df = df[df["original_label"].notna() & (df["original_label"] != "tool_misuse")].copy()
    excluded = before - len(df)
    df["label_id"] = df["original_label"].map(LABEL2ID)
    assert df["label_id"].isna().sum() == 0, f"Unmapped labels in {split_name}: {df['original_label'].unique()}"
    df["label_id"] = df["label_id"].astype(int)
    print(f"[{split_name}] kept {len(df)} / {before} rows ({excluded} excluded: tool_misuse or blind-NaN)")
    print(f"  class counts: {df['original_label'].value_counts().to_dict()}")
    return df

train_df = load_intent_split(LAYER1_TRAIN_PATH, "train")
val_df   = load_intent_split(LAYER1_VAL_PATH,   "validation")
test_df  = load_intent_split(LAYER1_TEST_PATH,  "test")   # AAAC-only after filtering — see markdown above

# Supplemental MSA slice — dev_osact has real (non-blind) safe/hate_speech labels
dev_osact = pd.read_csv(DEV_OSACT_PATH)
dev_osact = dev_osact[dev_osact["layer1_label"].notna()].copy()
dev_osact["original_label"] = dev_osact["layer1_label"]
dev_osact["label_id"] = dev_osact["original_label"].map(LABEL2ID)
dev_osact = dev_osact[dev_osact["label_id"].notna()].copy()
dev_osact["label_id"] = dev_osact["label_id"].astype(int)
dev_osact["language_variant"] = "osact_msa"
print(f"[dev_osact supplemental] {len(dev_osact)} labeled MSA rows: {dev_osact['original_label'].value_counts().to_dict()}")

text_col = "text_clean" if "text_clean" in train_df.columns else "text"
print("Using text column:", text_col)

def fill_text_fallback(df, primary_col, fallback_col="text"):
    """AAAC rows never populate text_clean because AAAC's raw `text` is already
    clean (LLM-generated red-teaming data — no @USER/<LF>/hashtag noise). Only
    OSACT (real scraped tweets) needed a separate cleaning pass, which is why
    text_clean exists at all. This is NOT a data-quality issue, just a column
    that was never filled for a source that never needed it — verified: 0/2472
    AAAC train rows contain any OSACT-style noise marker (@USER, <LF>, RT, URL).
    Falls back to raw `text` for those rows with zero information loss."""
    if primary_col == fallback_col or fallback_col not in df.columns:
        return df
    is_bad = df[primary_col].isna() | (df[primary_col].astype(str).str.strip() == "")
    n_filled = 0
    if is_bad.any():
        can_fill = is_bad & df[fallback_col].notna() & (df[fallback_col].astype(str).str.strip() != "")
        df.loc[can_fill, primary_col] = df.loc[can_fill, fallback_col]
        n_filled = int(can_fill.sum())
    still_bad = df[primary_col].isna() | (df[primary_col].astype(str).str.strip() == "")
    print(f"  filled {n_filled} rows from '{fallback_col}' (AAAC rows, already clean), "
          f"{int(still_bad.sum())} still empty (dropped)")
    return df[~still_bad].copy()

print("Applying text_clean fallback (fills AAAC rows from raw `text`, which is already clean):")
train_df = fill_text_fallback(train_df, text_col)
val_df   = fill_text_fallback(val_df, text_col)
test_df  = fill_text_fallback(test_df, text_col)
# dev_osact is 100% OSACT-sourced (real tweets), so text_clean should already be fully
# populated there — this call is just a defensive no-op guard, not expected to fill anything.
if text_col in dev_osact.columns:
    dev_osact = fill_text_fallback(dev_osact, text_col)

## 4. Sample Weights (alpha) + Focal Loss (gamma) — class balance + real-MSA difficulty for `hate_speech`

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

OSACT_HATE_SPEECH_BOOST = 1.35   # alpha multiplier for hate_speech rows with source == "OSACT"
FOCAL_GAMMA = 2.0                # standard focal-loss focusing parameter (Lin et al. 2017 default)
USE_FOCAL_LOSS = True            # set False to fall back to plain alpha-weighted CE (Part 6)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_LABELS),
    y=train_df["label_id"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("Base class weights (safe/hate_speech/prompt_injection/sensitive_request):", class_weights.round(3))

def compute_sample_weights(df, class_weights, boost_label="hate_speech", boost_source="OSACT",
                            boost_factor=OSACT_HATE_SPEECH_BOOST):
    """Per-example alpha weight = base class weight, with an extra multiplier for the
    specific (label, source) combination that's underperforming on real data.
    Requires a `source` column (present in layer1_train/validation.csv — verified
    'OSACT' / 'AAAC' values). Combined with focal loss's (1-p_t)^gamma term below,
    not used alone anymore — see markdown above for why."""
    w = df["label_id"].map(lambda i: class_weights[i]).astype(float).values.copy()
    if "source" in df.columns:
        boost_mask = (df["original_label"].values == boost_label) & (df["source"].values == boost_source)
        w[boost_mask] *= boost_factor
        print(f"  boosted {boost_mask.sum()} '{boost_label}' rows with source='{boost_source}' "
              f"by {boost_factor}x")
    else:
        print("  WARNING: no 'source' column found — sample weights fall back to class-level only.")
    return w

train_sample_weights = compute_sample_weights(train_df, class_weights)
val_sample_weights   = compute_sample_weights(val_df, class_weights)   # used only for logging/consistency; loss uses train weights


## 5. PyTorch Dataset (now carries per-example sample weights)

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, sample_weights=None, max_length=MAX_LENGTH):
        self.encodings = tokenizer(list(texts), truncation=True, padding="max_length", max_length=max_length)
        self.labels = list(labels)
        self.sample_weights = list(sample_weights) if sample_weights is not None else None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        if self.sample_weights is not None:
            item["sample_weight"] = torch.tensor(self.sample_weights[idx], dtype=torch.float32)
        return item


## 6. Weighted-Loss Trainer, Metrics, and Shared Inference Helper

In [ ]:
class WeightedLossTrainer(Trainer):
    """Extends 05a's pattern. Three loss modes, in priority order:
      1. sample_weight present + USE_FOCAL_LOSS=True  -> alpha-balanced focal loss
      2. sample_weight present + USE_FOCAL_LOSS=False -> plain alpha-weighted CE (first-fix behavior)
      3. no sample_weight                              -> single per-class weight tensor (05a's pattern)
    """
    def __init__(self, *args, class_weights_tensor=None, use_focal_loss=True, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor
        self.use_focal_loss = use_focal_loss
        self.focal_gamma = focal_gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        sample_weight = inputs.pop("sample_weight", None)
        outputs = model(**inputs)
        logits = outputs.logits
        num_labels = model.config.num_labels

        per_example_ce = torch.nn.functional.cross_entropy(
            logits.view(-1, num_labels), labels.view(-1), reduction="none")

        if sample_weight is not None:
            alpha = sample_weight.to(logits.device)
            if self.use_focal_loss:
                p_t = torch.exp(-per_example_ce)   # softmax prob of the true class
                focal_term = (1.0 - p_t).clamp(min=0.0).pow(self.focal_gamma)
                per_example_loss = alpha * focal_term * per_example_ce
            else:
                per_example_loss = alpha * per_example_ce
            loss = per_example_loss.sum() / alpha.sum().clamp(min=1e-8)
        else:
            weight = (self.class_weights_tensor.to(logits.device)
                      if self.class_weights_tensor is not None else None)
            loss = torch.nn.CrossEntropyLoss(weight=weight)(
                logits.view(-1, num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    _, _, macro_f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_f1": macro_f1}


def run_inference(df, tokenizer, model, text_column):
    model.eval()
    texts = list(df[text_column])
    all_logits = []
    batch_size = 32
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
            logits = model(**enc).logits
            all_logits.append(logits.cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)
    return preds, probs


def dev_osact_hate_speech_f1(tokenizer, model):
    """Used inside the seed-selection loop (Part 7) — see composite-metric rationale."""
    preds, _ = run_inference(dev_osact, tokenizer, model, text_col)
    y_true = dev_osact["label_id"].values
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        y_true, preds, labels=[LABEL2ID["safe"], LABEL2ID["hate_speech"]], zero_division=0)
    return float(f1_per_class[1])   # index 1 = hate_speech


## 7. Multi-Seed Training (3 seeds — same as 05a)

In [ ]:
SEED_SELECTION_VAL_WEIGHT = 0.6
SEED_SELECTION_DEV_OSACT_WEIGHT = 0.4

def train_intent_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    print(f"\n{'='*70}\n Training Intent Agent — seed={seed}\n{'='*70}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
    ).to(DEVICE)

    train_ds = TextClassificationDataset(train_df[text_col], train_df["label_id"], tokenizer,
                                          sample_weights=train_sample_weights)
    val_ds   = TextClassificationDataset(val_df[text_col],   val_df["label_id"],   tokenizer,
                                          sample_weights=val_sample_weights)

    args = TrainingArguments(
        output_dir=f"{OUT_DIR}/tmp_seed{seed}",
        num_train_epochs=6,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
    )

    trainer = WeightedLossTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        class_weights_tensor=class_weights_tensor,   # fallback only — sample_weight in batch takes precedence
        use_focal_loss=USE_FOCAL_LOSS,
        focal_gamma=FOCAL_GAMMA,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    val_metrics = trainer.evaluate()

    dev_osact_f1 = dev_osact_hate_speech_f1(tokenizer, trainer.model)

    return trainer, tokenizer, val_metrics, train_time, dev_osact_f1


seed_results = {}
for seed in SEEDS:
    trainer, tokenizer, val_metrics, train_time, dev_osact_f1 = train_intent_seed(seed)
    composite = (SEED_SELECTION_VAL_WEIGHT * val_metrics["eval_macro_f1"]
                 + SEED_SELECTION_DEV_OSACT_WEIGHT * dev_osact_f1)
    seed_results[seed] = {
        "trainer": trainer, "tokenizer": tokenizer,
        "val_macro_f1": val_metrics["eval_macro_f1"],
        "val_accuracy": val_metrics["eval_accuracy"],
        "dev_osact_hate_speech_f1": dev_osact_f1,
        "composite_score": composite,
        "train_time_s": train_time,
    }
    print(f"[seed={seed}] val_macro_f1={val_metrics['eval_macro_f1']:.4f}  "
          f"dev_osact_hate_speech_f1={dev_osact_f1:.4f}  composite={composite:.4f}  "
          f"train_time={train_time:.0f}s")

best_seed = max(seed_results, key=lambda s: seed_results[s]["composite_score"])
composite_values = [seed_results[s]["composite_score"] for s in SEEDS]
print(f"\nBest seed: {best_seed}  (composite={seed_results[best_seed]['composite_score']:.4f}, "
      f"val_macro_f1={seed_results[best_seed]['val_macro_f1']:.4f}, "
      f"dev_osact_hate_speech_f1={seed_results[best_seed]['dev_osact_hate_speech_f1']:.4f})")
print(f"Composite-score variance across seeds: mean={np.mean(composite_values):.4f}  "
      f"std={np.std(composite_values):.4f}")

best_trainer   = seed_results[best_seed]["trainer"]
best_tokenizer = seed_results[best_seed]["tokenizer"]
best_model     = best_trainer.model


## 8. Evaluation — AAAC-derived test set + `dev_osact` supplemental MSA slice

In [ ]:
def evaluate_split(df, tokenizer, model, name, classes_present=None):
    preds, probs = run_inference(df, tokenizer, model, text_col)
    y_true = df["label_id"].values
    labels_present = classes_present if classes_present is not None else sorted(set(y_true) | set(preds))
    target_names = [ID2LABEL[i] for i in labels_present]

    report = classification_report(y_true, preds, labels=labels_present, target_names=target_names,
                                    output_dict=True, zero_division=0)
    print(f"\n=== {name} ===")
    print(classification_report(y_true, preds, labels=labels_present, target_names=target_names, zero_division=0))

    cm = confusion_matrix(y_true, preds, labels=labels_present)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Intent Agent — {name}")
    plt.tight_layout()
    fig.savefig(f"{REPORT_DIR}/confusion_matrix_{name.replace(' ','_').lower()}.png", dpi=300)
    plt.show()

    return preds, probs, report, cm


test_preds, test_probs, test_report, test_cm = evaluate_split(test_df, best_tokenizer, best_model, "AAAC test (dialects)")
osact_preds, osact_probs, osact_report, osact_cm = evaluate_split(
    dev_osact, best_tokenizer, best_model, "dev_osact (MSA, safe+hate_speech only)",
    classes_present=[LABEL2ID["safe"], LABEL2ID["hate_speech"]],
)


## 9. Per-Language-Variant Robustness (Wilson CI for small n)

In [ ]:
def wilson_ci(successes, n, alpha=0.05):
    if n == 0:
        return (float("nan"), float("nan"))
    return proportion_confint(successes, n, alpha=alpha, method="wilson")

lang_rows = []
for variant, sub in test_df.groupby("language_variant"):
    idx = sub.index
    pos = np.array([i in idx for i in test_df.index])
    y_true_v = test_df.loc[pos, "label_id"].values
    y_pred_v = test_preds[pos.values] if hasattr(pos, "values") else test_preds[pos]
    correct = (y_true_v == y_pred_v)
    lo, hi = wilson_ci(int(correct.sum()), len(correct))
    lang_rows.append({"language_variant": variant, "n": len(correct),
                       "accuracy": correct.mean(), "ci_low": lo, "ci_high": hi})

lang_df = pd.DataFrame(lang_rows)
lang_df.to_csv(f"{REPORT_DIR}/per_language_robustness.csv", index=False)
display(lang_df)

fig, ax = plt.subplots(figsize=(7, 4.5))
# Calculate lower and upper error bar lengths, ensuring they are non-negative
lower_error = np.maximum(0, lang_df["accuracy"] - lang_df["ci_low"])
upper_error = np.maximum(0, lang_df["ci_high"] - lang_df["accuracy"])

ax.bar(lang_df["language_variant"], lang_df["accuracy"],
       yerr=[lower_error, upper_error],
       capsize=4)
ax.set_ylabel("Accuracy (95% Wilson CI)"); ax.set_title("Intent Agent — accuracy by language variant")
plt.tight_layout()
fig.savefig(f"{REPORT_DIR}/per_language_robustness.png", dpi=300)
plt.show()

## 10. Threshold Calibration

In [ ]:
import sys
sys.path.insert(0, OUT_DIR)   # expects threshold_calibration.py copied here — see setup note below
# If threshold_calibration.py isn't already on Drive, upload it once to OUT_DIR before running this cell.
from threshold_calibration import (calibrate_conf_margin_threshold, calibrate_ood_entropy_threshold,apply_manual_override_if_degenerate, export_threshold_json, batch_entropy)


MANUAL_FALLBACK = {
    "SENSITIVE_MIN_CONF": 0.65, "SENSITIVE_MIN_MARGIN": 0.20,
    "INJECTION_MIN_CONF": 0.65, "INJECTION_MIN_MARGIN": 0.20,
}

val_preds, val_probs = run_inference(val_df, best_tokenizer, best_model, text_col)
val_conf = val_probs.max(axis=1)
sorted_probs = np.sort(val_probs, axis=1)[:, ::-1]
val_margin = sorted_probs[:, 0] - sorted_probs[:, 1]
val_entropy = batch_entropy(val_probs)
val_is_correct = (val_preds == val_df["label_id"].values)

# sensitive_request gate (R6) — sweep, then override if degenerate
is_sensitive_pred = (val_preds == LABEL2ID["sensitive_request"])
y_true_sensitive = (val_df["label_id"].values == LABEL2ID["sensitive_request"]).astype(int)
sens_conf_res, sens_margin_res, sens_sweep_df = calibrate_conf_margin_threshold(
    y_true_sensitive, is_sensitive_pred, val_conf, val_margin, threshold_name="SENSITIVE")
sens_conf_res = apply_manual_override_if_degenerate(
    sens_conf_res, MANUAL_FALLBACK["SENSITIVE_MIN_CONF"],
    "only 37 validation positives, model already near-perfect (AAAC test F1=0.94) — no reliable interior optimum exists")
sens_margin_res = apply_manual_override_if_degenerate(
    sens_margin_res, MANUAL_FALLBACK["SENSITIVE_MIN_MARGIN"],
    "same as SENSITIVE_MIN_CONF — see note above")

# prompt_injection gate (R5) — sweep, then override if degenerate
is_injection_pred = (val_preds == LABEL2ID["prompt_injection"])
y_true_injection = (val_df["label_id"].values == LABEL2ID["prompt_injection"]).astype(int)
inj_conf_res, inj_margin_res, inj_sweep_df = calibrate_conf_margin_threshold(
    y_true_injection, is_injection_pred, val_conf, val_margin, threshold_name="INJECTION")
inj_conf_res = apply_manual_override_if_degenerate(
    inj_conf_res, MANUAL_FALLBACK["INJECTION_MIN_CONF"],
    "only 49 validation positives, model already near-perfect (AAAC test F1=0.99) — no reliable interior optimum exists")
inj_margin_res = apply_manual_override_if_degenerate(
    inj_margin_res, MANUAL_FALLBACK["INJECTION_MIN_MARGIN"],
    "same as INJECTION_MIN_CONF — see note above")

# OOD entropy gate — combined val + dev_osact pool (real fix, not another floor raise)
dev_osact_preds, dev_osact_probs = run_inference(dev_osact, best_tokenizer, best_model, text_col)
dev_osact_entropy = batch_entropy(dev_osact_probs)
dev_osact_is_correct = (dev_osact_preds == dev_osact["label_id"].values)

combined_entropy = np.concatenate([val_entropy, dev_osact_entropy])
combined_is_correct = np.concatenate([val_is_correct, dev_osact_is_correct])
print(f"Entropy calibration pool: {len(val_entropy)} (val) + {len(dev_osact_entropy)} (dev_osact) "
      f"= {len(combined_entropy)} total, {int((~combined_is_correct).sum())} wrong "
      f"(was {int((~val_is_correct).sum())} wrong using val alone)")

ood_res, ood_sweep_df = calibrate_ood_entropy_threshold(combined_entropy, combined_is_correct, n_classes=NUM_LABELS)

all_results = {
    "SENSITIVE_MIN_CONF": sens_conf_res, "SENSITIVE_MIN_MARGIN": sens_margin_res,
    "INJECTION_MIN_CONF": inj_conf_res, "INJECTION_MIN_MARGIN": inj_margin_res,
    "OOD_ENTROPY_THRESHOLD": ood_res,
}
print("=" * 78)
for name, res in all_results.items():
    flag = "  \u26A0 FLAGGED" if ("DEGENERATE" in res.notes or "UNRELIABLE" in res.notes) else (
           "  \u2713 MANUAL" if "MANUAL_OVERRIDE" in res.notes else "  OK")
    print(f"{name:<24} = {res.chosen_value:<8}{flag}")
    if res.notes:
        print(f"    -> {res.notes}")
print("=" * 78)

export_threshold_json(
    f"{MODEL_OUT}/threshold.json",
    all_results,
    extra_meta={"calibrated_on": "layer1_validation.csv (SENSITIVE/INJECTION gates) + "
                                  "layer1_validation.csv+dev_osact.csv combined pool (OOD entropy gate)",
                "n_validation": int(len(val_df)), "n_dev_osact": int(len(dev_osact)),
                "backbone": MODEL_CHECKPOINT, "num_labels": NUM_LABELS},
)
print(f"\nthreshold.json written to {MODEL_OUT}/threshold.json")
print("MANUAL lines are stated judgment calls (sweep couldn't constrain them), not sweep results —")
print("cite them as such in the paper rather than presenting every number as empirically tuned.")

# Save sweep curves for the paper
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df_sweep) in zip(axes, [("sensitive_request", sens_sweep_df),
                                        ("prompt_injection", inj_sweep_df)] + [("ood_entropy", ood_sweep_df)]):
    if "axis" in df_sweep.columns:
        for axis_name, g in df_sweep.groupby("axis"):
            ax.plot(g["value"], g["f1"], label=axis_name)
        ax.legend(); ax.set_title(f"{name} threshold sweep")
    else:
        ax.plot(df_sweep["threshold"], df_sweep["youden_j"])
        ax.set_title("OOD entropy sweep (Youden's J)")
plt.tight_layout()
fig.savefig(f"{REPORT_DIR}/threshold_sweep_curves.png", dpi=300)
plt.show()


## 11. Save Model, Label Map, Metrics

In [ ]:
best_model.save_pretrained(MODEL_OUT)
best_tokenizer.save_pretrained(MODEL_OUT)

with open(f"{MODEL_OUT}/label_map.json", "w", encoding="utf-8") as f:
    json.dump({"id2label": {str(k): v for k, v in ID2LABEL.items()},
               "label2id": LABEL2ID}, f, ensure_ascii=False, indent=2)

metrics_summary = {
    "backbone": MODEL_CHECKPOINT,
    "num_labels": NUM_LABELS,
    "labels": list(ID2LABEL.values()),
    "excluded_labels": ["tool_misuse"],
    "osact_hate_speech_boost_factor": OSACT_HATE_SPEECH_BOOST,
    "seed_selection_weights": {"val_macro_f1": SEED_SELECTION_VAL_WEIGHT,
                               "dev_osact_hate_speech_f1": SEED_SELECTION_DEV_OSACT_WEIGHT},
    "best_seed": best_seed,
    "seed_scores": {str(s): {"val_macro_f1": seed_results[s]["val_macro_f1"],
                              "dev_osact_hate_speech_f1": seed_results[s]["dev_osact_hate_speech_f1"],
                              "composite_score": seed_results[s]["composite_score"]} for s in SEEDS},
    "composite_score_variance_std": float(np.std(composite_values)),
    "test_aaac_dialects": test_report,
    "test_dev_osact_msa": osact_report,
    "per_language_robustness": lang_df.to_dict(orient="records"),
    "trained_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(f"{MODEL_OUT}/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2, default=str)

print("Saved to:", MODEL_OUT)
print("Files:", os.listdir(MODEL_OUT))


In [ ]:
# Add inside compute_loss, right after computing focal_term, temporarily:
if self.state.global_step < 5:
    print(f"[FOCAL CHECK] use_focal_loss={self.use_focal_loss} | "
          f"p_t sample={p_t[:3].tolist()} | focal_term sample={focal_term[:3].tolist()}")